# HCMAIC P3 keyframe runner

Notebook này không chứa bản sao thuật toán. Nó clone đúng branch HCMAIC và gọi cùng CLI với local để tránh hai implementation bị lệch.

Trước khi chạy: bật **GPU** và **Internet**, attach dataset video cùng file weight TransNetV2, và bảo đảm branch cấu hình bên dưới đã được push.

In [ ]:
REPO_URL = "https://github.com/iamtro-2006/event-retrieval-system.git"
BRANCH = "UpdateKeyframeP3"
GROUPS = ["Videos_L21_a", "Videos_L22_a"]
VIDEO_IDS = []  # Rỗng = chạy toàn bộ video trong các GROUPS ở trên
DATASET_ROOT = None  # Để None để tự tìm thư mục cha của GROUPS

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

working = Path("/kaggle/working")
repo = working / "event-retrieval-system"
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "open_clip_torch==3.3.0", "decord==0.6.0", "ffmpeg-python==0.2.0", "PyYAML>=6.0"], check=True)

In [ ]:
import cv2
import open_clip
import torch
import yaml

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Kaggle Settings, select a GPU accelerator and restart the session.")
if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg executable is unavailable in this Kaggle image.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| OpenCV:", cv2.__version__, "| OpenCLIP:", open_clip.__version__)

In [ ]:
input_root = Path("/kaggle/input")
if not GROUPS and not DATASET_ROOT:
    raise ValueError("Set GROUPS or provide DATASET_ROOT before running the notebook.")
if DATASET_ROOT:
    dataset_root = Path(DATASET_ROOT)
else:
    group_dirs = []
    for group in GROUPS:
        hits = [path for path in input_root.rglob(group) if path.is_dir()]
        if len(hits) != 1:
            raise RuntimeError(f"Expected one Kaggle input directory for {group}, found: {hits}")
        group_dirs.append(hits[0])
    parents = {path.parent.resolve() for path in group_dirs}
    if len(parents) != 1:
        raise RuntimeError(f"All requested groups must share one dataset root, found: {parents}")
    dataset_root = parents.pop()

weights = list(input_root.rglob("transnetv2-pytorch-weights.pth"))
if not weights:
    raise FileNotFoundError("Attach a Kaggle Dataset containing transnetv2-pytorch-weights.pth")
print("Dataset root:", dataset_root)
print("Groups:", GROUPS)
print("TransNet weights:", weights[0])

In [ ]:
backend = repo / "backend"
source_config = backend / "configs" / "kf_extraction.yaml"
runtime_config = working / "kf_extraction_kaggle.yaml"
with source_config.open(encoding="utf-8") as stream:
    config = yaml.safe_load(stream)
config["paths"]["input_dir"] = str(dataset_root)
config["paths"]["output_dir"] = str(working / "processed")
config["paths"]["cache_dir"] = str(working / ".cache" / "keyframe_extraction")
config["transnet"]["weights_path"] = str(weights[0])
config["transnet"]["device"] = "cuda"
config["keyframe"]["selection_embedding"]["device"] = "cuda"
config["keyframe"]["write_diagnostics"] = False
config["logging"]["log_to_file"] = False
with runtime_config.open("w", encoding="utf-8") as stream:
    yaml.safe_dump(config, stream, sort_keys=False)

command = [sys.executable, "scripts/keyframe_extraction/run.py", "--config", str(runtime_config)]
if GROUPS:
    command += ["--groups", *GROUPS]
if VIDEO_IDS:
    command += ["--video-ids", *VIDEO_IDS]
subprocess.run(command, cwd=backend, check=True)

In [ ]:
output = Path("/kaggle/working/processed")
images = list((output / "keyframes").rglob("*.jpg"))
maps = list((output / "map_keyframes").rglob("*.csv"))
print(f"Completed: {len(images)} keyframe images, {len(maps)} mapping files")
print("Kaggle output:", output)